In [ ]:
"""
train_and_predict.py

Requirements:
  pip install pandas scikit-learn joblib
"""

import os
import re
import joblib
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ---------------------------
# Config
# ---------------------------
DATA_PATH = "complaints_dataset.csv"   # change to your CSV file
MODEL_PATH = "priority_model.joblib"
RANDOM_STATE = 42
TEST_SIZE = 0.20

# ---------------------------
# Helpers
# ---------------------------
def clean_text(text: str) -> str:
    """Basic text cleaning: lower, remove extra spaces and non-alphanum (keep spaces)."""
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_dataset(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {"text", "location", "duration_days", "priority"}
    if not required.issubset(df.columns):
        raise ValueError(f"Dataset must contain columns: {required}")
    # Ensure types
    df = df.copy()
    df['text'] = df['text'].astype(str).apply(clean_text)
    df['location'] = df['location'].astype(str)
    df['duration_days'] = pd.to_numeric(df['duration_days'], errors='coerce').fillna(0).astype(int)
    df['priority'] = df['priority'].astype(str)
    return df

# ---------------------------
# Train / Build pipeline
# ---------------------------
def build_pipeline():
    # Column transformer: text -> tfidf, location -> onehot, duration -> scaler
    preprocessor = ColumnTransformer(
        transformers=[
            ("txt", TfidfVectorizer(ngram_range=(1, 2), max_features=2000), "text"),
            ("loc", OneHotEncoder(handle_unknown="ignore"), ["location"]),
            ("dur", StandardScaler(), ["duration_days"]),
        ],
        remainder="drop"
    )

    clf = Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))
    ])
    return clf

# ---------------------------
# Train and evaluate
# ---------------------------
def train_and_evaluate(df: pd.DataFrame, model_path: str = MODEL_PATH):
    # Encode labels
    le = LabelEncoder()
    y = le.fit_transform(df['priority'])
    X = df[['text', 'location', 'duration_days']]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y if len(set(y))>1 else None
    )

    # Build and train
    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    # Evaluate
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0, target_names=le.classes_)

    print(f"Test accuracy: {acc:.4f}")
    print("Classification report:")
    print(report)

    # Save both pipeline and label encoder together
    joblib.dump({"pipeline": pipeline, "label_encoder": le}, model_path)
    print(f"Saved model -> {model_path}")
    return pipeline, le

# ---------------------------
# Load model and predict
# ---------------------------
def load_model(model_path: str = MODEL_PATH):
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Model file not found at {model_path}. Train first.")
    data = joblib.load(model_path)
    return data['pipeline'], data['label_encoder']

def predict_priority(complaint: str, location: str = "residential", duration: int = 1, model_tuple=None):
    """
    Predict priority. Provide either model_tuple=(pipeline, label_encoder) or ensure model saved at MODEL_PATH.
    Returns human-readable label (e.g., 'High').
    """
    if not isinstance(complaint, str) or len(complaint.strip().split()) < 2:
        return "Invalid complaint"

    if not isinstance(duration, int) or duration < 0:
        return "Invalid duration"

    complaint_clean = clean_text(complaint)

    if model_tuple is None:
        pipeline, le = load_model()
    else:
        pipeline, le = model_tuple

    X = pd.DataFrame([{
        "text": complaint_clean,
        "location": location,
        "duration_days": duration
    }])

    y_pred_enc = pipeline.predict(X)[0]
    y_pred = le.inverse_transform([int(y_pred_enc)])[0]
    return y_pred

# ---------------------------
# Main: train if data present, else show usage
# ---------------------------
if __name__ == "__main__":
    if Path(DATA_PATH).exists():
        print("Loading dataset:", DATA_PATH)
        df = load_dataset(DATA_PATH)
        pipeline, le = train_and_evaluate(df, MODEL_PATH)
    else:
        print(f"No dataset found at {DATA_PATH}. Please place your CSV there.")
        print("CSV required columns: text, location, duration_days, priority")
        raise SystemExit(1)

    # CLI example for interactive prediction
    print("\n--- Interactive prediction ---")
    print("Available locations:", sorted(df['location'].unique()))
    user_text = input("Enter complaint description: ").strip()
    user_loc = input("Enter location (leave blank for 'residential'): ").strip() or "residential"
    try:
        user_dur = int(input("Duration in days (number, default 1): ").strip() or "1")
    except ValueError:
        print("Invalid duration. Using 1 day.")
        user_dur = 1

    pred = predict_priority(user_text, user_loc, user_dur, model_tuple=(pipeline, le))
    print(f"Predicted priority: {pred}")


Loading dataset: complaints_dataset.csv
Test accuracy: 0.5000
Classification report:
              precision    recall  f1-score   support

        High       0.67      1.00      0.80         2
         Low       0.00      0.00      0.00         1
      Medium       0.00      0.00      0.00         1

    accuracy                           0.50         4
   macro avg       0.22      0.33      0.27         4
weighted avg       0.33      0.50      0.40         4

Saved model -> priority_model.joblib

--- Interactive prediction ---
Available locations: ['commercial', 'construction_site', 'downtown', 'intersection', 'main_road', 'municipal', 'park', 'residential', 'sidewalk', 'streetlight']
